# ΔF/F with a robust multi-exponential baseline

`fmproc.dff` subtracts a fitted baseline $F_0$ from each fiber's fluorescence and divides by it:
`ΔF/F = (F - F0) / F0`. It is built from two steps that can also be used on their own: an estimator,
`fmproc.multiexp_baseline`, returns $F_0$ as a recording, and `fmproc.baseline_correction` combines a
signal with its baseline (`(F - F0) / F0`, or `F - F0`). $F_0$ is a sum of 2-3 bleaching exponentials (plus an
optional brightening term), fitted with a Tukey biweight so calcium transients don't pull the baseline up.
The fit runs once, when the baseline is built, and its parameters are stored with the recording.

They also take a `FiberPhotometryRecordingGroup` (`baseline_correction` the signal group and its baseline
group): each color is fitted on its own and the result is a group with the same names.

It needs the `aind-ophys-utils` package: `pip install -e .[dff]`.

This notebook simulates what we typically record, two fibers in green, iso and red, with known baselines
and known transients, so the result can be checked against the truth. The baselines are fits to a real
85-minute photometry session.

In [ ]:
%matplotlib inline
import json
import pathlib
import tempfile
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from aind_ophys_utils.dff import plot_dff
from spikeinterface.core import load
from spikeinterface.preprocessing import apply_preprocessing_pipeline

import fiber_mosaic.processing as fmproc
from fiber_mosaic import FiberPhotometryRecordingGroup
from fiber_mosaic.core.base import recording_from_traces
from fiber_mosaic.generation import calcium_transients
from fiber_mosaic.processing.pipeline import FiberPreprocessingPipeline

plt.rcParams["axes.xmargin"] = 0.01  # lines run (almost) edge to edge

# one meaning per color in every figure
COLORS = {"green": "#009E73", "iso": "#0072B2", "red": "#D55E00"}
F0_COLOR = "#F0E442"  # the fitted baseline
TRUTH_COLOR = "k"  # the simulated truth


def plot_f0(ax, t, f, f0_hat, color, f0_true=None, title=""):
    """Plot F with its fitted (and, if given, true) baseline."""
    ax.plot(t, f, lw=0.4, color=color, label="F")
    ax.plot(t, f0_hat, lw=2, color=F0_COLOR, label=r"fitted F$_0$")
    if f0_true is not None:
        style = {"lw": 0.8, "ls": "--", "color": TRUTH_COLOR}
        ax.plot(t, f0_true, label=r"true F$_0$", **style)
    ax.set(title=title, ylabel="F (a.u.)")

## 1. Simulate

The baselines are bleaching fits to the two dLight fibers of a typical 85-minute session (the slow time
constants are as long as the recording or longer; a negative amplitude is a brightening term, which
mostly cancels a slower positive exponential). Only green carries sensor transients (about 40 % ΔF/F,
20 per minute, decay 0.25 s). Noise is white with an SD of 1.4 % in green, 0.81 % in iso and 1.0 % in red
(measured on first differences across sessions).

In [ ]:
FS = 20.0
n_samples = 102_000  # 85 minutes
times = np.arange(n_samples) / FS

# fits to a real session, per color and fiber:
# (b_inf, [(amplitude, tau in s), ...])
BASELINES = {
    "green": [
        (1421.0, [(616.0, 7696.0), (132.0, 1573.0), (95.8, 103.0)]),
        (186.0, [(990.0, 32607.0), (46.6, 476.0)]),
    ],
    "iso": [
        (747.0, [(1302.0, 18409.0), (91.0, 251.0)]),
        (134.0, [(1401.0, 33579.0), (1351.0, 533.0), (-1313.0, 582.0)]),
    ],
    "red": [
        (293.0, [(1005.0, 49879.0), (1005.0, 4475.0), (-851.0, 3059.0)]),
        (690.0, [(434.0, 6491.0), (217.0, 718.0), (-434.0, 1143.0)]),
    ],
}
# (transient amplitude, relative white-noise SD)
SENSOR = {
    "green": (0.4, 0.014),
    "iso": (0.0, 0.0081),
    "red": (0.0, 0.0100),
}


def simulate(seed, color, b_inf, terms):
    """Return (true F0, true ΔF/F, noisy trace) for one fiber."""
    amplitude, noise_sd = SENSOR[color]
    rng = np.random.default_rng(seed)
    f0 = b_inf + sum(a * np.exp(-times / tau) for a, tau in terms)
    calcium = calcium_transients(
        times, rng, num_events=n_samples // 60, decay=0.25, amplitude=amplitude
    )
    noise = rng.normal(0.0, noise_sd, n_samples)
    return f0, calcium, f0 * (1 + calcium + noise)


f0_true, calcium_true, traces = {}, {}, {}
for i, (color, fits) in enumerate(BASELINES.items()):
    runs = [
        simulate(10 * i + fiber, color, *fit) for fiber, fit in enumerate(fits)
    ]
    f0_true[color], calcium_true[color], noisy = (
        np.stack(r, axis=1) for r in zip(*runs, strict=True)
    )
    traces[color] = noisy.astype("float32")

group = FiberPhotometryRecordingGroup(
    {
        color: recording_from_traces(
            traces[color], name=color, sampling_frequency=FS
        )
        for color in BASELINES
    }
)
group

## 2. Run the step

`multiexp_baseline` fits every fiber of every recording (about a second each) and returns the baseline $F_0$
as a recording of the same shape. `baseline_correction` combines a signal with its baseline: the default
`mode="dff"` gives `(F - F0) / F0`, `mode="df"` gives `F - F0`. `fmproc.dff` is the one-call shortcut for the two
and also works as a pipeline step (see section 3).

In [ ]:
start = time.perf_counter()
baseline = fmproc.multiexp_baseline(group)
elapsed = time.perf_counter() - start
print(
    f"fitted {len(group)} recordings x {group.get_num_fibers()} fibers "
    f"in {elapsed:.1f} s"
)
dff = fmproc.baseline_correction(group, baseline)
dff

In [ ]:
result = {
    color: rec.get_traces() for color, rec in dff.items()
}  # ΔF/F, (samples, fibers)
f0_est = {
    color: rec.get_traces() for color, rec in baseline.items()
}  # the fitted baseline

fig, axes = plt.subplots(3, 2, figsize=(13, 8), sharex=True)
for row, color in enumerate(group):
    for fiber in range(2):
        plot_f0(
            axes[row, fiber],
            times / 60,
            traces[color][:, fiber],
            f0_est[color][:, fiber],
            COLORS[color],
            f0_true[color][:, fiber],
            title=f"{color}, fiber {fiber}",
        )
axes[0, 0].legend(loc="upper right")
for ax in axes[-1]:
    ax.set_xlabel("time (min)")
for ax in axes[:, 1]:
    ax.set_ylabel("")  # same label as the left column
fig.tight_layout()

In [ ]:
early = times < 120
for color in group:
    for fiber in range(2):
        error = np.abs(f0_est[color][:, fiber] / f0_true[color][:, fiber] - 1)
        line = (
            f"{color:5s} fiber {fiber}: "
            f"median |F0 error| {100 * np.median(error):.2f} %, "
            f"first 2 min {100 * np.median(error[early]):.2f} %"
        )
        if color == "green":
            corr = np.corrcoef(
                result[color][:, fiber], calcium_true[color][:, fiber]
            )[0, 1]
            line += f", corr(ΔF/F, true ΔF/F) {corr:.3f}"
        print(line)

In [ ]:
# F with its baseline, and ΔF/F with zooms on the start, middle and end
fig = plot_dff(
    traces["green"][:, 0],
    f0_est["green"][:, 0],
    times,
    title="green, fiber 0",
    color=COLORS["green"],
)

Green ΔF/F follows the true transients (correlation 0.99). The median baseline error is 0.3 % in green, which
carries the transients, and 0.01 % in iso and red fiber 0, whose brightening terms are recovered too. Red fiber 1
is the weak spot: its cancelling brightening and exponential terms are fitted with other time constants, which
leaves a bias of about 1 % in the first 2 minutes. The fitted parameters are not the simulated ones: green fiber 0
is fitted with two exponentials instead of three (see section 5). Terms with similar time constants trade off,
and only their sum, the baseline, is identifiable.

## 3. As a pipeline step

`dff` is registered by name, so it can sit in a `FiberPreprocessingPipeline` like any other step, and
the pipeline takes the group as well. Step options go under `"params"`. The pipeline fits again, so this
takes a few seconds too.

In [ ]:
pipeline = FiberPreprocessingPipeline([{"name": "dff", "params": {"ds": 10}}])
via_pipeline = apply_preprocessing_pipeline(group, pipeline)
for color in group:
    np.testing.assert_allclose(
        via_pipeline[color].get_traces(), result[color], atol=1e-5
    )
via_pipeline

## 4. Non-finite samples

NaN or inf samples are left out of the fit with a warning and remain non-finite in the ΔF/F (NaN stays NaN,
±inf stays ±inf). The fitted baseline is a smooth curve, so it bridges a gap. Here the green fiber 0 recording
drops out for 10 minutes.

In [ ]:
gap = np.zeros(n_samples, dtype=bool)
gap[30 * 60 * int(FS) : 40 * 60 * int(FS)] = True  # minutes 30 to 40
damaged = traces["green"].copy()
damaged[gap, 0] = np.nan
damaged_rec = recording_from_traces(
    damaged, name="green", sampling_frequency=FS
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    damaged_f0 = fmproc.multiexp_baseline(damaged_rec)
print("warning:", caught[0].message)
damaged_dff = fmproc.baseline_correction(damaged_rec, damaged_f0)

f0_hat = damaged_f0.get_traces()[:, 0]
error = np.abs(f0_hat / f0_true["green"][:, 0] - 1)
print(
    "ΔF/F is NaN exactly in the gap:",
    np.array_equal(np.isnan(damaged_dff.get_traces()[:, 0]), gap),
)
print(
    f"median |F0 error|: {100 * np.median(error[gap]):.2f} % in the gap, "
    f"{100 * np.median(error[~gap]):.2f} % elsewhere"
)

window = (times > 25 * 60) & (times < 45 * 60)
fig, ax = plt.subplots(figsize=(13, 3))
plot_f0(
    ax,
    times[window] / 60,
    damaged[window, 0],
    f0_hat[window],
    COLORS["green"],
    title="green, fiber 0, dropout from minute 30 to 40",
)
ax.set_xlabel("time (min)")
ax.legend(loc="upper right")
plt.show()

## 5. Saving and reloading

The fitted parameters live in the baseline recording's kwargs, so a saved correction reloads **without
refitting**. Shown for the green recording. Unused slots (here the third exponential and the brightening
term) are padded with amplitude 0 and tau = inf; the infinities are written as `null`, so the JSON stays standard.

In [ ]:
green = group["green"]
folder = pathlib.Path(tempfile.mkdtemp())
# arrays can't be dumped to JSON, so write the traces to disk first
with warnings.catch_warnings():
    # SpikeInterface: in-memory provenance isn't saved
    warnings.simplefilter("ignore")
    on_disk = green.save(folder=folder / "traces", progress_bar=False)
saved = fmproc.dff(on_disk)
saved.dump_to_json(folder / "dff.json")

document = json.loads((folder / "dff.json").read_text())
print("fiber 0: [b_inf, b1, tau1, b2, tau2, b3, tau3, b_bright, tau_bright]")
print(
    document["kwargs"]["baseline_recording"]["kwargs"]["baseline_params"][0][0]
)

In [ ]:
start = time.perf_counter()
reloaded = load(folder / "dff.json")
same = np.array_equal(reloaded.get_traces(), saved.get_traces())
elapsed = time.perf_counter() - start
print(f"reloaded in {elapsed:.2f} s, identical to the original: {same}")

## Options

`fmproc.multiexp_baseline(recording, rss_thresh=(0.98, 0.97), maxiter=8, ds=10, noise_method="mad")`; `fmproc.dff`
passes these on. See `fiber_mosaic.processing.baseline.fit_baseline` for what each does. `ds` is the
decimation used while choosing the model; the final fit always uses every sample.